# Afternoon class 30/08 — Worksheet 04 SOLUTIONS: loc and iloc   (L01/L02)

Every cell below was executed in the lab image (pandas 3.0.5) and the quoted
output is what it actually printed — including the error in Q10.

Questions 6 and 10 are the ones to re-read. Q6 is a rule nobody mentions until
it bites; Q10 is a line printed on a lecture slide that cannot run.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 04 — loc and iloc. Run this once.
import pandas as pd

# The lecture's frame: labels S1..S3, and only TWO columns.
marks = pd.DataFrame(
    {"Name": ["Sara", "Ahmed", "Lina"], "Marks": [85, 90, 75]},
    index=["S1", "S2", "S3"],
)

# The same data on a DEFAULT index, where labels and positions look alike.
plain = marks.reset_index(drop=True)

# An index of integers running BACKWARDS, so that no label sits at the
# position that shares its number. That is what makes loc and iloc diverge.
shuffled = pd.DataFrame(
    {"Name": ["Sara", "Ahmed", "Lina"], "Marks": [85, 90, 75]},
    index=[3, 2, 1],
)

print(marks)
print()
print("columns:", list(marks.columns), "-> there are only two")

PART A — the rule

### Question 1

`loc["S1"]` and `iloc[0]` -> the same row. `loc["S1", "Marks"]` and `iloc[0, 1]` -> both `85`.

Two spellings of the same address, and here they agree. Note the second
argument in each: `.loc` takes the column *name*, `.iloc` takes the column
*number*. The comma separates rows from columns in both.

Everything for the rest of this sheet is about the cases where the two
spellings stop agreeing.

In [ ]:
print("marks.loc['S1']:")
print(marks.loc["S1"])
print()
print("marks.iloc[0]:")
print(marks.iloc[0])
print()
print("one cell by label:   ", marks.loc["S1", "Marks"])
print("one cell by position:", marks.iloc[0, 1])

### Question 2

`plain.loc[0]` and `plain.iloc[0]` -> the same row, Sara.

They agree because `plain`'s labels are `0, 1, 2` — so 'the row labelled
0' and 'the row in position 0' happen to name the same row.

This is the coincidence that makes the whole distinction feel academic.
Most frames you meet have a default index, so most of the time you can use
either and never find out which one you meant. Q3 removes the
coincidence.

In [ ]:
print(plain)
print()
print("plain.loc[0]:")
print(plain.loc[0])
print()
print("plain.iloc[0]:")
print(plain.iloc[0])

# They agree only because the labels happen to be 0,1,2 -- so "the label 0"
# and "position 0" name the same row. Nothing guarantees that.

### Question 3

`shuffled.loc[1]` -> **Lina**. `shuffled.iloc[1]` -> **Ahmed**.

The index runs `[3, 2, 1]`, so label `1` sits in position 2 and position 1
holds label `2`. Same expression as Q2, same integer, different student.

Neither call raised. Both returned a plausible row. If you had used the
wrong one inside a loop over student IDs, you would get a complete,
well-formed, entirely wrong answer — no traceback, no warning, nothing to
investigate.

Indexes like this are not exotic. Any frame that has been filtered,
sorted, or concatenated has an index whose numbers no longer match its
positions. `reset_index(drop=True)` is what restores the coincidence, and
it is worth doing deliberately rather than assuming.

In [ ]:
print(shuffled)
print()
print("shuffled.loc[1]  -> the row LABELLED 1:")
print(shuffled.loc[1])
print()
print("shuffled.iloc[1] -> the row in POSITION 1:")
print(shuffled.iloc[1])

PART B — selecting more than one thing

### Question 4

`loc[["S1","S3"]]` and `iloc[[0,2]]` -> the same two rows. Column both ways -> `[85, 90, 75]`.

Lists work in both, and `:` means 'all of this axis' in both. So
`marks.loc[:, "Marks"]` reads as 'every row, the Marks column'.

Worth noticing that the selected rows kept their labels `S1` and `S3` —
they were not renumbered to `0` and `1`. Selection never renumbers, which
is the thing Q8 depends on.

In [ ]:
print("two rows by label:")
print(marks.loc[["S1", "S3"]])
print()
print("two rows by position:")
print(marks.iloc[[0, 2]])
print()
print("one whole column by label:   ", list(marks.loc[:, "Marks"]))
print("one whole column by position:", list(marks.iloc[:, 1]))

### Question 5

`loc["S1":"S2"]` -> 2 rows. `iloc[0:2]` -> 2 rows.

Both returned two rows, so nothing looks wrong yet. Hold the numbers: two
slices that look parallel gave the same count.

They are not parallel, and Q6 is the same comparison written so the
difference is visible.

In [ ]:
print("marks.loc['S1':'S2']:")
print(marks.loc["S1":"S2"])
print("-> rows:", len(marks.loc["S1":"S2"]))
print()
print("marks.iloc[0:2]:")
print(marks.iloc[0:2])
print("-> rows:", len(marks.iloc[0:2]))

### Question 6

`loc["S1":"S3"]` -> **3 rows**. `iloc[0:2]` -> **2 rows**. -> `iloc[0:3]` is what returns all three.

Both slices named the same third row as their endpoint — `'S3'` is its
label, `2` is its position — and they disagreed about whether to include
it.

**`.loc` slices include the end label. `.iloc` slices exclude the end
position**, like every other Python slice. So to reach the last row with
`.iloc` you must go one past it.

Neither deck mentions this, and it is the most common way to lose exactly
one row. The failure is silent: `iloc[0:2]` on a three-row frame returns a
perfectly valid two-row frame. If those were the rows of a report, you
shipped it missing its last line and nothing anywhere said so.

The reason `.loc` behaves this way is that labels have no arithmetic —
there is no 'one past `S3`' to name, so an exclusive endpoint would make
label slices unable to reach their own last element.

In [ ]:
# Both slices name the SAME third row as their endpoint:
#   'S3' is the label of the third row, 2 is its position.
print("loc['S1':'S3'] ->", len(marks.loc["S1":"S3"]), "rows:", list(marks.loc["S1":"S3"].index))
print("iloc[0:2]      ->", len(marks.iloc[0:2]), "rows:", list(marks.iloc[0:2].index))
print()
# To get all three rows from iloc you have to go one PAST the last position:
print("iloc[0:3]      ->", len(marks.iloc[0:3]), "rows:", list(marks.iloc[0:3].index))

# .loc slices are INCLUSIVE of the end label.
# .iloc slices are EXCLUSIVE of the end position, like every other Python slice.

PART C — filtering

### Question 7

All three forms -> Sara `85` and Ahmed `90`. -> `loc[mask, "Name"]` gives just the two names.

`marks[mask]` and `marks.loc[mask]` are the same operation; the bare form
is shorthand. The `.loc` form is worth the extra characters because it
takes a second argument, so you can filter rows and pick columns in one
step instead of chaining two selections.

That also matters for assignment: `marks.loc[mask, "Marks"] = 0` is the
supported way to write into a filtered subset. Chained selections like
`marks[mask]["Marks"] = 0` write into a temporary and are silently
discarded.

In [ ]:
print(marks[marks["Marks"] > 80])
print()
print(marks.loc[marks["Marks"] > 80])
print()
print("just the names:")
print(marks.loc[marks["Marks"] > 80, "Name"])

### Question 8

Filtered index -> `['S1', 'S2']`. `top.iloc[-1]` -> **Ahmed**, `marks.iloc[-1]` -> **Lina**.

The filtered frame kept the labels `S1` and `S2` — filtering removes rows,
it never renumbers the survivors.

So positions shifted and labels did not. `.iloc[0]` agrees across both
frames by luck; `.iloc[-1]` does not, because the last row of the filtered
frame is Ahmed while the last row of the original is Lina.

This is why 'get the last row' is a fragile instruction. On a filtered
frame, position `-1` means 'last survivor', which is a different question
from 'last record' — and the two only coincide when nothing was removed.

In [ ]:
top = marks[marks["Marks"] > 80]
print(top)
print("index kept:", list(top.index))
print()
print("top.iloc[0]   is", top.iloc[0]["Name"])
print("marks.iloc[0] is", marks.iloc[0]["Name"])
print()
# Same position, different frames, and here they happen to agree. Now the
# other end:
print("top.iloc[-1]   is", top.iloc[-1]["Name"])
print("marks.iloc[-1] is", marks.iloc[-1]["Name"])

### Question 9

`loc['S2','Marks']` -> `90` (Ahmed). `iloc[0, 1]` -> `85` (Sara). `iloc[1, 1]` -> `90` (Ahmed).

Three addresses, two of which reach Ahmed and one of which reaches Sara.
The summary table on the L02 deck writes `df.iloc[0, 1]` for its worked
example, and on this frame that is Sara's mark, not Ahmed's.

The mapping is worth writing out once: `iloc[0, 1]` is row 0 (`S1`, Sara),
column 1 (`Marks`). Nothing in the expression contains the word 'Sara' or
'Marks', which is exactly why position-based addressing is so easy to get
wrong and so hard to review.

In [ ]:
print("marks.loc['S2','Marks'] ->", marks.loc["S2", "Marks"], "(Ahmed)")
print("marks.iloc[0, 1]        ->", marks.iloc[0, 1], "(Sara)")
print("marks.iloc[1, 1]        ->", marks.iloc[1, 1], "(Ahmed)")
print()
print(marks)

### Question 10

`marks.iloc[0, 2]` -> **raises** `IndexError: index 2 is out of bounds for axis 0 with size 2`.

The slide prints this exact expression and gives the answer as `85`. It
cannot produce `85`, or anything else — the frame has two columns, so the
only valid column positions are `0` and `1`.

And the very next slide's summary table writes the same idea as
`df.iloc[0, 1]`, which does work and does give `85`. Two consecutive
slides, one of which cannot run. When a deck disagrees with itself, run
both versions — that takes ten seconds and settles it permanently.

One oddity in the message: it says **`axis 0`** even though you indexed out
of bounds on the columns. Pandas has handed the lookup down to the
underlying NumPy machinery for a single axis at a time, and the message is
reported in those terms. Do not let it send you looking at your rows.

In [ ]:
print("columns:", list(marks.columns), "-> positions 0 and 1 exist, 2 does not")
print(marks.iloc[0, 2])